In [1]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0,org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,org.apache.iceberg:iceberg-aws-bundle:1.10.0 pyspark-shell'
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, BooleanType, TimestampType
)

In [2]:
spark = (
    SparkSession.builder
    .appName("silver_cdc")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [3]:
CUSTOMERS_SCHEMA = StructType([
    StructField("id",         IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("email",      StringType(),  True),
    StructField("country",    StringType(),  True),
    StructField("created_at", StringType(),  True),
])

DRIVERS_SCHEMA = StructType([
    StructField("id",             IntegerType(), True),
    StructField("name",           StringType(),  True),
    StructField("license_number", StringType(),  True),
    StructField("rating",         StringType(),  True),
    StructField("city",           StringType(),  True),
    StructField("active",         BooleanType(), True),
    StructField("created_at",     StringType(),  True),
])

STATE_FILE       = "/home/jovyan/checkpoints/silver_cdc_state.json"
BRONZE_TABLE     = "lakehouse.cdc.bronze"
SILVER_CUSTOMERS = "lakehouse.cdc.silver_customers"
SILVER_DRIVERS   = "lakehouse.cdc.silver_drivers"

In [4]:
def ensure_tables(spark):
    spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER_CUSTOMERS} (
            id         INT,
            name       STRING,
            email      STRING,
            country    STRING,
            created_at STRING
        )
        USING iceberg
        TBLPROPERTIES ('write.upsert.enabled' = 'true')
    """)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER_DRIVERS} (
            id             INT,
            name           STRING,
            license_number STRING,
            rating         STRING,
            city           STRING,
            active         BOOLEAN,
            created_at     STRING
        )
        USING iceberg
        TBLPROPERTIES ('write.upsert.enabled' = 'true')
    """)
    print("[silver_cdc] Silver tables ready.")

In [5]:
def load_state() -> dict:
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f:
            return json.load(f)
    return {}


def save_state(state: dict):
    os.makedirs(os.path.dirname(STATE_FILE), exist_ok=True)
    with open(STATE_FILE, "w") as f:
        json.dump(state, f)

In [6]:
def get_latest_snapshot(spark) -> int | None:
    rows = spark.sql(f"""
        SELECT snapshot_id
        FROM {BRONZE_TABLE}.snapshots
        ORDER BY committed_at DESC
        LIMIT 1
    """).collect()
    return rows[0]["snapshot_id"] if rows else None

In [7]:
def read_bronze_incremental(spark, state: dict):
    latest_snapshot = get_latest_snapshot(spark)
    if latest_snapshot is None:
        print("[silver_cdc] Bronze table has no snapshots yet.")
        return None, None

    last_snapshot = state.get("last_bronze_snapshot")

    if last_snapshot is None:
        print("[silver_cdc] First run — reading all bronze rows.")
        df = spark.read.format("iceberg").load(BRONZE_TABLE)
    elif last_snapshot == latest_snapshot:
        print("[silver_cdc] No new bronze snapshots since last run. Nothing to do.")
        return None, latest_snapshot
    else:
        print(f"[silver_cdc] Incremental read: after snapshot {last_snapshot} → up to {latest_snapshot}")
        df = (
            spark.read
            .format("iceberg")
            .option("start-snapshot-id", last_snapshot)
            .load(BRONZE_TABLE)
        )

    return df, latest_snapshot

In [8]:
def apply_merge(spark, bronze_df, table_name: str, schema: StructType, source_table_name: str):
    from pyspark.sql.window import Window

    df = bronze_df.filter(F.col("source_table") == source_table_name)
    count = df.count()
    if count == 0:
        print(f"[silver_cdc] No new events for {source_table_name}.")
        return

    after_parsed  = F.from_json(F.col("after_json"),  schema)
    before_parsed = F.from_json(F.col("before_json"), schema)

    df = df.select(
        F.col("op"),
        F.col("ts_ms"),
        after_parsed.alias("after"),
        before_parsed.alias("before"),
    ).withColumn(
        "record_id",
        F.when(F.col("op") == "d", F.col("before.id")).otherwise(F.col("after.id"))
    )

    window = Window.partitionBy("record_id").orderBy(F.col("ts_ms").desc())
    deduped = (
        df.withColumn("rn", F.row_number().over(window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    n_del    = deduped.filter(F.col("op") == "d").count()
    n_upsert = deduped.count() - n_del
    print(f"[silver_cdc] {source_table_name}: {n_upsert} upserts + {n_del} deletes to apply")

    staging_table = f"lakehouse.cdc._staging_{source_table_name}"

    deduped.writeTo(staging_table).using("iceberg").createOrReplace()

    cols       = [f.name for f in schema.fields if f.name != "id"]
    set_clause = ", ".join(f"t.{c} = s.after.{c}" for c in cols)
    ins_cols   = ", ".join(["id"] + cols)
    ins_vals   = ", ".join(f"s.after.{c}" for c in ["id"] + cols)

    spark.sql(f"""
        MERGE INTO {table_name} t
        USING {staging_table} s
        ON t.id = s.record_id
        WHEN MATCHED AND s.op = 'd'
            THEN DELETE
        WHEN MATCHED AND s.op IN ('u', 'c', 'r')
            THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED AND s.op IN ('c', 'r')
            THEN INSERT ({ins_cols}) VALUES ({ins_vals})
    """)

    spark.sql(f"DROP TABLE IF EXISTS {staging_table}")

    after_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {table_name}").collect()[0]["cnt"]
    print(f"[silver_cdc] {table_name} → {after_count} rows after MERGE.")

In [9]:
ensure_tables(spark)

state = load_state()
bronze_df, latest_snapshot = read_bronze_incremental(spark, state)

if bronze_df is not None:
    apply_merge(spark, bronze_df, SILVER_CUSTOMERS, CUSTOMERS_SCHEMA, "customers")
    apply_merge(spark, bronze_df, SILVER_DRIVERS,   DRIVERS_SCHEMA,   "drivers")
    save_state({"last_bronze_snapshot": latest_snapshot})
    print(f"[silver_cdc] State saved. Last snapshot: {latest_snapshot}")
else:
    print("[silver_cdc] Nothing to process — silver already up to date.")

print("\n[silver_cdc] Snapshot history for silver_customers:")
spark.sql(f"""
    SELECT snapshot_id, committed_at, operation, summary
    FROM {SILVER_CUSTOMERS}.snapshots
    ORDER BY committed_at DESC
    LIMIT 10
""").show(truncate=False)

print("\n[silver_cdc] Snapshot history for silver_drivers:")
spark.sql(f"""
    SELECT snapshot_id, committed_at, operation, summary
    FROM {SILVER_DRIVERS}.snapshots
    ORDER BY committed_at DESC
    LIMIT 10
""").show(truncate=False)

[silver_cdc] Silver tables ready.
[silver_cdc] First run — reading all bronze rows.
[silver_cdc] customers: 10 upserts + 0 deletes to apply
[silver_cdc] lakehouse.cdc.silver_customers → 10 rows after MERGE.
[silver_cdc] drivers: 8 upserts + 0 deletes to apply
[silver_cdc] lakehouse.cdc.silver_drivers → 8 rows after MERGE.
[silver_cdc] State saved. Last snapshot: 7685603044006889168

[silver_cdc] Snapshot history for silver_customers:
+-------------------+----------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |committed_at          |op